# Creative OS — Node Engine Prototype

Implements the Hexagonal Blueprint node transition logic from `spec/hexagonal_blueprint_v04.md`.

**Rules enforced:**
- Simultaneous activation limit: max 3 non-floor nodes; exceeding without prior Z fires Lore Creep
- Floor nodes (π, project-elevated H) do not consume a slot
- A tracks per-entity; multiple instances count as one node type in the slot count
- Z fires in three modes: Complete, Truncated, Partial/External
- S deactivation: forcing function and merge-completion are separate events
- Post-S null-gap: logged when S deactivates with a shared goal still active (Ruling 6.6)

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


Z_MODES = {'Complete', 'Truncated', 'Partial/External'}


class NodeEngine:
    """Tracks node state against Hexagonal Blueprint rules."""

    def __init__(self, project: str, floor_nodes: Optional[set] = None):
        self.project = project
        self.floor_nodes = floor_nodes or set()  # nodes elevated per Ruling 6.1
        self.active: dict[str, list[str]] = {}   # node -> list of entity labels
        self.z_mode: Optional[str] = None
        self.deferred_z: list[dict] = []         # truncated/partial Z watch list
        self.post_s_gaps: list[dict] = []        # Ruling 6.6 null-gaps
        self.violations: list[dict] = []         # Lore Creep alerts
        self._log: list[dict] = []

    # ── Slot accounting ──────────────────────────────────────────────────────

    def slot_count(self) -> int:
        return sum(1 for n in self.active if n not in self.floor_nodes)

    def _active_display(self) -> list[str]:
        out = []
        for node, entities in self.active.items():
            tag = '[floor]' if node in self.floor_nodes else ''
            if node == 'A':
                out.extend(f'A({e})' for e in entities)
            else:
                out.append(f'{node}{tag}')
        return out

    # ── Activation ───────────────────────────────────────────────────────────

    def activate(self, node: str, scene: str, entity: str = '_',
                 z_mode: Optional[str] = None) -> None:
        # Slot limit: non-Z activations check before firing
        if node != 'Z' and node not in self.floor_nodes:
            if self.slot_count() >= 3:
                v = {
                    'type': 'LORE_CREEP',
                    'scene': scene, 'node': node,
                    'slot_count': self.slot_count(),
                    'active': self._active_display(),
                }
                self.violations.append(v)
                print(f'  ⚠  LORE_CREEP at {scene}: {node} would make slot_count='
                      f'{self.slot_count() + 1} without prior Z')

        if node not in self.active:
            self.active[node] = []
        if entity not in self.active[node]:
            self.active[node].append(entity)

        if node == 'Z':
            self.z_mode = z_mode or 'Complete'
            if self.z_mode in ('Truncated', 'Partial/External'):
                self.deferred_z.append({
                    'entity': entity, 'mode': self.z_mode, 'opened_at': scene
                })

        self._record(scene, 'ACTIVATE', node, entity, z_mode=z_mode)

    # ── Deactivation ─────────────────────────────────────────────────────────

    def deactivate(self, node: str, scene: str, entity: str = '_',
                   forcing_function: Optional[str] = None,
                   goal_active: bool = False) -> None:
        # Ruling 6.6: log null-gap when S deactivates with goal still active
        if node == 'S' and goal_active:
            gap = {
                'scene': scene,
                'forcing_function': forcing_function,
                'note': 'S deactivated — shared goal active, unified pursuit gap (Ruling 6.6)',
            }
            self.post_s_gaps.append(gap)
            print(f'  ◌  POST-S GAP at {scene}: unified pursuit untracked'
                  + (f' [forced by: {forcing_function}]' if forcing_function else ''))

        if node == 'Z':
            self.z_mode = None

        if node in self.active:
            if entity in self.active[node]:
                self.active[node].remove(entity)
            if not self.active[node]:
                del self.active[node]

        self._record(scene, 'DEACTIVATE', node, entity,
                     forcing_function=forcing_function)

    # ── Z deferred resolution ─────────────────────────────────────────────────

    def resolve_deferred_z(self, scene: str, entity: str = '_') -> None:
        """Call when a truncated/partial Z completes (Ruling 6.4 conditions met)."""
        still_open = [d for d in self.deferred_z if d['entity'] != entity]
        resolved = [d for d in self.deferred_z if d['entity'] == entity]
        self.deferred_z = still_open
        for r in resolved:
            print(f'  ✓  DEFERRED Z RESOLVED at {scene}: '
                  f"{r['entity']} ({r['mode']} from {r['opened_at']}) → Complete")
        self._record(scene, 'Z_RESOLVED', 'Z', entity)

    # ── Reporting ─────────────────────────────────────────────────────────────

    def report(self, scene: str) -> None:
        active = self._active_display()
        slots = self.slot_count()
        z_info = f' | Z:{self.z_mode}' if self.z_mode else ''
        print(f'  {scene:<45} active={active}  slots={slots}{z_info}')

    def _record(self, scene, action, node, entity,
                z_mode=None, forcing_function=None):
        self._log.append({
            'scene': scene, 'action': action, 'node': node, 'entity': entity,
            'z_mode': z_mode, 'forcing_function': forcing_function,
            'slot_count': self.slot_count(),
            'active': self._active_display(),
        })

    def summary(self) -> None:
        print(f'\n── {self.project} Summary ──')
        print(f'  Violations (Lore Creep): {len(self.violations)}')
        print(f'  Post-S gaps logged:      {len(self.post_s_gaps)}')
        print(f'  Deferred Z still open:   {len(self.deferred_z)}')
        if self.deferred_z:
            for d in self.deferred_z:
                print(f'    → {d["entity"]} ({d["mode"]} opened at {d["opened_at"]})')
        print(f'  Active nodes at close:   {self._active_display()}')
        print(f'  Slot count at close:     {self.slot_count()}')

## Veritas — Setup

H is elevated to floor status per Ruling 6.1 (the Hum never recedes — premise permanently dissolves structural bounds).

In [ ]:
# Veritas project: H on floor
eng = NodeEngine(project='Veritas', floor_nodes={'H'})

print('Project:', eng.project)
print('Floor nodes:', eng.floor_nodes)
print('π always present (not tracked as slot — structural floor)')

## Trace 001 — Prologue + Chapter 1

In [ ]:
print('── Prologue ──')
eng.activate('W', 'prologue.open')
eng.report('prologue.open')

# Hum arrives — H activates (elevates to floor), W deactivates
eng.activate('H', 'prologue.hum_arrives')  # no slot consumed — floor node
eng.deactivate('W', 'prologue.hum_arrives')  # active tension now detected
eng.report('prologue.hum_arrives')
# γ fires (generative — age of Veritas begins)

print('\n── Chapter 1: Cassie thread ──')
# Dana kitchen scene — A activates (Truth vs. Lie in balance)
eng.activate('A', 'ch1.kitchen_dana', entity='Cassie_domestic')
eng.report('ch1.kitchen_dana')  # slots: A=1

# Car ride — A holds, σ elevated on path
eng.report('ch1.car_ride')

# Boardroom — Z fires (Complete) before slot limit reached
eng.activate('Z', 'ch1.boardroom', z_mode='Complete')  # A + Z = 2 slots
eng.report('ch1.boardroom')
# α, δ, τ fire (duration-dependent — boardroom required full scene)

eng.deactivate('Z', 'ch1.boardroom_close')  # Z deactivates, narrative back in motion
eng.report('ch1.boardroom_close')

print('\n── Chapter 1: Valeria thread ──')
# γ fires (new thread — Valeria introduced)
# A(Valeria) activates — truth vs. necessary lie
eng.activate('A', 'ch1.precinct_intro', entity='Valeria')
eng.report('ch1.precinct_intro')  # A(Cassie_domestic) + A(Valeria) = 1 slot

# Reyes confession — A(precinct) deactivates, A(Valeria) persists
eng.deactivate('A', 'ch1.reyes_confession', entity='Cassie_domestic')
# Note: Cassie_domestic was the stand-in for the precinct's institutional A
# In this trace, precinct A was bundled with Cassie thread — refine in v2
eng.report('ch1.reyes_confession')

# Chapter 1 close — α fires toward Ch.2
eng.report('ch1.close')
print('\nTrace 001 complete.')

## Trace 002 — Chapter 2

In [ ]:
print('── Chapter 2: Cassie thread ──')

# Apartment — A(Cassie+Dana) activates, Z fires (Complete — personal coherence check)
eng.activate('A', 'ch2.apartment_dana', entity='Cassie+Dana')
eng.activate('Z', 'ch2.apartment_z', z_mode='Complete')  # A(Valeria)+A(Cassie+Dana)+Z
eng.report('ch2.apartment_z')  # slots: A=1 (two instances), Z=1 → total 2

# Confession complete — vase shatters, A(Cassie+Dana) resolves, Z deactivates
eng.deactivate('A', 'ch2.vase_shatters', entity='Cassie+Dana')
eng.deactivate('Z', 'ch2.vase_shatters')
eng.report('ch2.vase_shatters')  # slots: A(Valeria)=1

# Office — no Z (no false self remaining to confirm)
# α blocked (spin attempt fails), δ fires when she runs
eng.report('ch2.office_collapse')

print('\n── Chapter 2: Valeria thread ──')

# Precinct — Vance confession
# α, δ fire (cash hits floor, precinct state permanently shifted)
eng.report('ch2.precinct_vance')

# Chapter 2 close — α fires for S (convergence initiated by Valeria reading the text)
eng.activate('S', 'ch2.close_text_received')  # S in transition (α fired)
eng.report('ch2.close_text_received')  # slots: A(Valeria)=1, S=1 → total 2

print('\nTrace 002 complete.')

## Trace 003 — Chapter 3

In [ ]:
print('── Chapter 3: Drive → convergence ──')

# Drive — S in transition, σ at peak, τ fires (duration-dependent approach sequence)
eng.report('ch3.drive_truth_mob')  # slots: A(Valeria)=1, S=1 → 2

# Cassie enters bank — S fully activates (δ fires for S)
# (S was already activated in Ch.2 — δ fires here as completion)
eng.report('ch3.cassie_enters_bank')  # slots: A(Valeria)=1, S=1 → 2

# Confrontation — Z fires (Truncated: mob will interrupt before it completes)
eng.activate('Z', 'ch3.confrontation_gun_drawn', z_mode='Truncated')
eng.report('ch3.confrontation_gun_drawn')  # A + S + Z = 3 — AT LIMIT

# Mob invades — Z truncated (cut off), S deactivates (merge complete)
# Ruling 6.3: mob is forcing function; merge-completion is the cause
eng.deactivate('Z', 'ch3.mob_invades')  # Z truncated — deferred Z watch opened
eng.deactivate('S', 'ch3.mob_invades',
               forcing_function='Purified mob',
               goal_active=True)  # goal (Shadow Ledger) still active → null-gap logged
eng.report('ch3.mob_invades')  # slots: A(Valeria)=1

# Smoke grenade — δ fires (Valeria's purpose shifts: arrest → collaborate)
eng.report('ch3.smoke_grenade')

# Tunnel — A(Cassie+Valeria) activates (TILTED — Valeria's armor intact)
# Cassie calls out Valeria on the car. Valeria gets the last word.
eng.activate('A', 'ch3.tunnel_exchange', entity='Cassie+Valeria_tilted')
eng.report('ch3.tunnel_exchange')  # A(Valeria) + A(Cassie+Valeria) = 1 slot

# Chapter close — α fires toward Ch.4, darkness
eng.report('ch3.close_darkness')

print('\nTrace 003 complete.')

## Summary

In [ ]:
eng.summary()

print('\n── Deferred Z watch ──')
print('  Truncated Z opened at ch3.confrontation_gun_drawn')
print('  Resolution expected: Ch.8 sealed scene (Valeria self-confessed)')
print('  Condition: sealed scene + self-confessed + no interruption available')

print('\n── Ruling 6.6 validation ──')
for gap in eng.post_s_gaps:
    print(f"  Gap at {gap['scene']}: {gap['note']}")
    if gap['forcing_function']:
        print(f"  Forcing function (separate from cause): {gap['forcing_function']}")

print('\n── 3-node limit ──')
peak = max((e['slot_count'] for e in eng._log), default=0)
print(f'  Peak slot count across all traces: {peak}')
print(f'  Lore Creep violations: {len(eng.violations)}')
if not eng.violations:
    print('  ✓ Limit respected across all 3 chapters')